Volatility analysis measures the rate and magnitude of an asset's price fluctuations, quantifying risk and uncertainty in financial markets. It identifies rapid, unexpected price swings, helping investors distinguish between stable and high-risk assets

What is Volatility (in your context)

In finance (part of financial analysis):

Daily Return = % change of price
Volatility = standard deviation of returns

👉 Higher volatility = more risk
👉 Lower volatility = more stable stock

In [4]:
"""
Stock Analysis Script (SMA + EMA + Signals + Volatility)
"""

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import os
from pathlib import Path
import itertools

# ─────────────────────────────────────────────
# 1. CONFIGURATION
# ─────────────────────────────────────────────
CSV_FILES = {
    "HLL":      r"E:\KEC TASK\Download data\HLL Historical Data.csv",
    "TCS":      r"E:\KEC TASK\Download data\TCS Historical Data.csv",
    "RELIANCE": r"E:\KEC TASK\Download data\RELI Historical Data.csv",
    "MSFT":     r"E:\KEC TASK\Download data\MSFT Historical Data.csv",
    "HDBK":     r"E:\KEC TASK\Download data\HDBK Historical Data (1).csv",
}

OUTPUT_DIR = "output_charts_violity"
os.makedirs(OUTPUT_DIR, exist_ok=True)

colors = itertools.cycle(["#2196F3", "#FF5722", "#4CAF50", "#9C27B0", "#FF9800"])

# ─────────────────────────────────────────────
# 2. HELPER FUNCTIONS
# ─────────────────────────────────────────────
def parse_volume(vol_str):
    vol_str = str(vol_str).strip().upper().replace(",", "")
    if vol_str in ("", "N/A", "NAN", "-"):
        return np.nan
    multipliers = {"K": 1e3, "M": 1e6, "B": 1e9}
    for suffix, mult in multipliers.items():
        if vol_str.endswith(suffix):
            return float(vol_str[:-1]) * mult
    return float(vol_str)


def load_stock(filepath, ticker):
    if not Path(filepath).exists():
        print(f"[SKIP] {ticker} file not found")
        return None

    try:
        df = pd.read_csv(filepath)
        df.columns = [c.strip().strip('"') for c in df.columns]

        df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
        df = df.dropna(subset=["Date"]).sort_values("Date")

        for col in ["Price", "Open", "High", "Low"]:
            if col in df.columns:
                df[col] = df[col].astype(str).str.replace(",", "").astype(float)

        if "Vol." in df.columns:
            df["Volume"] = df["Vol."].apply(parse_volume)

        if "Change %" in df.columns:
            df["Change_pct"] = df["Change %"].astype(str).str.replace("%", "").astype(float)

        df["Ticker"] = ticker
        print(f"[OK] {ticker}: {len(df)} rows")
        return df

    except Exception as e:
        print(f"[ERROR] {ticker}: {e}")
        return None


# ─────────────────────────────────────────────
# 3. LOAD DATA
# ─────────────────────────────────────────────
stocks = {}
for ticker, path in CSV_FILES.items():
    df = load_stock(path, ticker)
    if df is not None:
        stocks[ticker] = df

if not stocks:
    raise SystemExit("No data loaded")

# ─────────────────────────────────────────────
# 4. INDICATORS (SMA + EMA + SIGNALS)
# ─────────────────────────────────────────────
for ticker, df in stocks.items():
    # SMA
    df["SMA5"] = df["Price"].rolling(5).mean()
    df["SMA10"] = df["Price"].rolling(10).mean()
    df["SMA20"] = df["Price"].rolling(20).mean()

    # EMA
    df["EMA5"] = df["Price"].ewm(span=5, adjust=False).mean()
    df["EMA10"] = df["Price"].ewm(span=10, adjust=False).mean()
    df["EMA20"] = df["Price"].ewm(span=20, adjust=False).mean()

    # Signals
    df["Signal"] = np.where(df["EMA5"] > df["EMA10"], 1, 0)
    df["Crossover"] = df["Signal"].diff()

# ─────────────────────────────────────────────
# 5. VOLATILITY ANALYSIS
# ─────────────────────────────────────────────
volatility_summary = []

for ticker, df in stocks.items():
    df["Daily_Return"] = df["Price"].pct_change()

    df["Volatility_5"] = df["Daily_Return"].rolling(5).std()
    df["Volatility_10"] = df["Daily_Return"].rolling(10).std()
    df["Volatility_20"] = df["Daily_Return"].rolling(20).std()

    df["Volatility_Annual"] = df["Daily_Return"].std() * np.sqrt(252)

    volatility_summary.append({
        "Ticker": ticker,
        "Avg Return (%)": round(df["Daily_Return"].mean() * 100, 3),
        "Daily Volatility": round(df["Daily_Return"].std(), 5),
        "Annual Volatility (%)": round(df["Volatility_Annual"].iloc[-1] * 100, 2),
    })

vol_df = pd.DataFrame(volatility_summary).set_index("Ticker")
vol_df.to_csv(os.path.join(OUTPUT_DIR, "volatility_summary.csv"))

print("\n📊 Volatility Summary:")
print(vol_df)

# ─────────────────────────────────────────────
# 6. PRICE + SMA + EMA CHART
# ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))

for (ticker, df), color in zip(stocks.items(), colors):
    ax.plot(df["Date"], df["Price"], label=f"{ticker} Price", color=color)
    ax.plot(df["Date"], df["SMA10"], linestyle="--", alpha=0.6, color=color)
    ax.plot(df["Date"], df["EMA10"], linestyle="-.", linewidth=1.5, color=color)

ax.set_title("Price with SMA & EMA")
ax.legend()
ax.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "price_sma_ema.png"))
plt.close()

# ─────────────────────────────────────────────
# 7. VOLATILITY CHART
# ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))

for (ticker, df), color in zip(stocks.items(), colors):
    ax.plot(df["Date"], df["Volatility_10"], label=ticker, color=color)

ax.set_title("Rolling Volatility (10-Day)")
ax.legend()
ax.grid(True)

plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "volatility_trend.png"))
plt.close()

# ─────────────────────────────────────────────
# 8. INDIVIDUAL STOCK CHARTS
# ─────────────────────────────────────────────
for (ticker, df), color in zip(stocks.items(), colors):

    # SIGNAL CHART
    fig, ax = plt.subplots(figsize=(12, 6))

    ax.plot(df["Date"], df["Price"], label="Price", color=color)
    ax.plot(df["Date"], df["EMA5"], label="EMA5")
    ax.plot(df["Date"], df["EMA10"], label="EMA10")

    buy = df[df["Crossover"] == 1]
    sell = df[df["Crossover"] == -1]

    ax.scatter(buy["Date"], buy["Price"], marker="^", color="green", s=80, label="BUY")
    ax.scatter(sell["Date"], sell["Price"], marker="v", color="red", s=80, label="SELL")

    ax.set_title(f"{ticker} Signals")
    ax.legend()
    ax.grid(True)

    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"{ticker}_signals.png"))
    plt.close()

    # VOLATILITY CHART
    fig, ax = plt.subplots(figsize=(12, 4))

    ax.plot(df["Date"], df["Volatility_10"], label="Volatility 10D", color="blue")
    ax.set_title(f"{ticker} Volatility")
    ax.legend()
    ax.grid(True)

    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"{ticker}_volatility.png"))
    plt.close()

# ─────────────────────────────────────────────
# 9. SAVE FINAL DATA
# ─────────────────────────────────────────────
all_data = pd.concat(stocks.values())
all_data.to_csv(os.path.join(OUTPUT_DIR, "violityanalysis_output.csv"), index=False)

print("\n✅ DONE — Full analysis (SMA + EMA + Signals + Volatility) completed!")

[OK] HLL: 20 rows
[OK] TCS: 20 rows
[OK] RELIANCE: 20 rows
[OK] MSFT: 21 rows
[OK] HDBK: 834 rows

📊 Volatility Summary:
          Avg Return (%)  Daily Volatility  Annual Volatility (%)
Ticker                                                           
HLL                0.647           0.01700                  26.98
TCS                0.252           0.02077                  32.98
RELIANCE           0.345           0.01942                  30.83
MSFT               0.915           0.01905                  30.23
HDBK               0.018           0.01583                  25.13

✅ DONE — Full analysis (SMA + EMA + Signals + Volatility) completed!


Data Loading & Cleaning


Feature Engineering (Core Logic)
Moving Averages
SMA (5, 10, 20 days)
EMA (5, 10, 20 days)



Trading Signals
Buy signal => EMA5 > EMA10
Sell signal => EMA5 < EMA10


eturns & Volatility
Daily return => percentage change
Rolling volatility (5, 10, 20 days)
Annual volatility


Analysis Output

Avg return
Daily volatility
Annual volatility


Output will Generated in "output_charts_violity" this  folder
│
├── price_sma_ema.png
├── volatility_trend.png
├── HLL_signals.png
├── HLL_volatility.png
├── volatility_summary.csv
└── violityanalysis_output.csv

